Quiz 52 — Stock Market Simulation  [SOLVED]
Difficulty: Hard

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(150)
means  = np.array([0.0005, 0.0008, 0.0002])
vols   = np.array([0.015,  0.025,  0.010])
names  = ["Stock X", "Stock Y", "Stock Z"]
days   = 252

# ── 1. Generate returns (252, 3) ──────────────────────────────────────────────
# Each column = one stock; each row = one day
returns = np.random.normal(means, vols, (days, 3))

# ── 2. Cumulative price ───────────────────────────────────────────────────────
# cumprod(1 + daily_return) builds compounding growth starting from 1.0
prices = np.cumprod(1 + returns, axis=0) * 100   # shape (252,3)

# ── 3. Final prices + returns ─────────────────────────────────────────────────
for i, name in enumerate(names):
    final   = prices[-1, i]
    pct_ret = (final - 100) / 100 * 100
    print(f"{name}: Final £{final:.2f}  |  Return: {pct_ret:+.2f}%")

# ── 4. Rolling max drawdown for Stock Y ──────────────────────────────────────
stock_y   = prices[:, 1]
worst_dd  = 0.0
worst_start = 0
for start in range(len(stock_y) - 30):
    window    = stock_y[start:start+30]
    roll_max  = np.maximum.accumulate(window)
    dd        = ((window - roll_max) / roll_max).min()
    if dd < worst_dd:
        worst_dd    = dd
        worst_start = start

print(f"\nStock Y worst 30-day drawdown: {worst_dd*100:.2f}% starting day {worst_start+1}")

# ── 5. VaR at 5% ─────────────────────────────────────────────────────────────
print("\nDaily VaR (5% level):")
for i, name in enumerate(names):
    var = np.percentile(returns[:, i], 5)
    print(f"  {name}: {var*100:.3f}%  (worst expected daily loss 95% of days)")

# ── 6. Plot ───────────────────────────────────────────────────────────────────
colours = ["steelblue","tomato","green"]
fig, ax = plt.subplots(figsize=(12, 6))
for i, (name, c) in enumerate(zip(names, colours)):
    ax.plot(prices[:, i], label=name, color=c)
ax.axhline(100, color="black", linestyle="--", linewidth=1, label="Start £100")
ax.set_title("Simulated Stock Prices — 1 Year (252 Days)")
ax.set_xlabel("Trading Day")
ax.set_ylabel("Price (£)")
ax.legend()
plt.tight_layout()
plt.savefig("hard_12_stock_simulation.png", dpi=100)
plt.show()

# ── WHY ───────────────────────────────────────────────────────────────────────
# np.cumprod(1 + returns, axis=0) applies compounding: each day's price is
# the previous day's price multiplied by (1 + daily return).
# VaR (Value at Risk) at 5% = the 5th percentile of daily returns —
# on 95% of days, the loss will not exceed this level.
# np.maximum.accumulate computes the running maximum without a loop.